# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirabdulbaqi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
%pip -q install duckdb huggingface_hub

In [11]:
import os
import getpass

# Read Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [12]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("✅ Connected to FlyRank Warehouse")

✅ Connected to FlyRank Warehouse


In [13]:
# Test the connection

con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


## 1. Unit of analysis + time window

For my lane, the unit of analysis is **one content page**. Each row represents the observed search performance of one content page during the selected analysis period. For this assignment, I will use a **mid-panel month (2026-03)** because it provides historical observations without using the final month as future information. My goal is to rank content pages based on their observed performance so the results can support content improvement decisions.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Features:** CTR, average position, impressions, clicks, engagement metrics, and other search performance measures that are available before making a ranking decision.

**Label / Proxy:** A content performance score or ranking based on observed search performance. This is the outcome the model will learn to predict.

**Context:** Client ID, Content ID, and report date. These fields help identify and organize records but should not be used as prediction features.

**Excluded:** Future information, manually assigned business decisions, and any columns derived from the target label. These are excluded because they would introduce data leakage and produce unrealistic model performance.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification Query 1: Row count and date range

query1 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
""").df()

print("Query 1: Row count and date range")
display(query1)

# Verification Query 2: Verify the grain

query2 = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS duplicates
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

print("Query 2: Grain check (should be empty)")
display(query2)

# Verification Query 3: Availability check

query3 = con.sql(f"""
SELECT
    COUNT(*) AS rows_with_gsc
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
""").df()

print("Query 3: GSC data available")
display(query3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1: Row count and date range


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 2: Grain check (should be empty)


,client_hash_id,content_hash_id,report_date,duplicates


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3: GSC data available


,rows_with_gsc
0,3611061


## 4. Data limits

This dataset supports decision-making based on observed search performance, but it cannot prove why content performance changed. External factors such as Google algorithm updates, seasonal trends, competitor actions, or marketing campaigns are not fully captured. The history is also unbalanced because different clients began collecting Google Search Console and Google Analytics data at different times. Therefore, the results should be interpreted as decision-support rather than causal evidence.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check the available data history for clients

con.sql(f"""
SELECT
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start
FROM {TABLES['dim_clients']}
""").df()


,earliest_gsc_start,latest_gsc_start
0,2025-01-27,2026-06-02


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.